In [ ]:
from enum import unique

import scanpy as sc
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import os
import numpy as np
from networkx.algorithms.bipartite.basic import color
#import pooch
from scipy.sparse import csr_matrix
from scipy.io import mmwrite
from pathlib import Path
import anndata
import copy
from scipy.io import mmread
from scipy.sparse import csr_matrix
import glob
import re

from sympy.physics.units import length
import snapatac2 as snap
snap.__version__
print("scanpy version:", sc.__version__)
print("numpy version:", np.__version__)

In [ ]:
os.getcwd()

In [ ]:
os.chdir('/mnt/d/lxk/project/jiangshucai20260506')

In [ ]:
import gzip
from pathlib import Path
import snapatac2 as snap

base_path = Path("./data/scatac/matrix")
output_dir = Path("./h5ad/scATAC")
output_dir.mkdir(parents=True, exist_ok=True)

sampleList = list()
with os.scandir(base_path) as entries:
    sampleid_list = [entry.name for entry in entries if entry.is_dir()]
sampleid_list

In [ ]:


adatas = []

for sample in sampleid_list:
    fragment_file = (
        base_path / sample / "output" / "fragments.tsv.gz"
    )
    barcode_file = (
        base_path / sample / "output"
        / "filter_peak_matrix" / "barcodes.tsv.gz"
    )
    output_file = output_dir / f"{sample}.h5ad"

    # 每个样本自己的有效细胞barcode
    with gzip.open(barcode_file, "rt") as file:
        whitelist = [line.strip() for line in file if line.strip()]

    print(f"{sample}: {len(whitelist):,}个有效barcode")

    adata = snap.pp.import_fragments(
        fragment_file,
        file=output_file,
        chrom_sizes=snap.genome.hg38,
        whitelist=whitelist,
        min_num_fragments=0,
        sorted_by_barcode=False,
        n_jobs = 20
    )

    adata.obs["sample"] = [sample] * adata.n_obs
    adatas.append(adata)

In [ ]:
%%time
for adata in adatas:
    snap.pp.add_tile_matrix(adata, bin_size=500)
    snap.pp.select_features(adata, n_features=200000)
    snap.pp.scrublet(adata)
    snap.pp.filter_doublets(adata)


In [ ]:
snap.metrics.tsse(
    adatas,
    gene_anno=snap.genome.hg38,
    n_jobs=7,
)

In [ ]:
for sample, adata in zip(sampleid_list, adatas):
    print(sample)

    snap.pl.tsse(
        adata,
        min_fragment=0,
        width=500,
        height=500,
        interactive=True,
        #out_file = f'./fig/atac/quality/{sample}.pdf'
    )

In [ ]:
snap.pp.filter_cells(adatas, min_counts=3000, min_tsse= 2, max_counts=100000)

# Creating AnnDataSet object

In [ ]:
dict_adatas = list(zip(sampleid_list, adatas))

In [ ]:
%%time
data = snap.AnnDataSet(
    adatas=dict_adatas,
    filename="./h5ad/atacAnndataSet.h5ads",
    add_key = 'sample' #将key添加到sample列，自动标明样本
)
data

In [ ]:
print(f'Number of cells: {data.n_obs}')
print(f'Number of unique barcodes: {np.unique(data.obs_names).size}')

## Resolve the conflict of obs_name

In [ ]:
unique_cell_ids = [sa + ':' + bc for sa, bc in zip(data.obs['sample'], data.obs_names)]
data.obs_names = unique_cell_ids
assert data.n_obs == np.unique(data.obs_names).size
snap.pp.select_features(data, n_features=200000)

In [ ]:
atac  = snap.pp.make_gene_matrix(data, gene_anno=snap.genome.hg38)

In [ ]:
data.close()

# 注释scATAC

## 1.准备rna数据

In [ ]:
rna = sc.read_h5ad("./h5ad/rna_0.h5ad")
rna.X = rna.layers["counts"].copy()

In [ ]:
print(rna.X.min(), rna.X.max())
print(atac.X.min(), atac.X.max())

## 2.提取共有基因

In [ ]:
shared_genes = rna.var_names.intersection(
    atac.var_names,
    sort=False,
)

print("共有基因数：", len(shared_genes))

rna = rna[:, shared_genes].copy()
atac = atac[:, shared_genes].copy()

In [ ]:
sc.pp.filter_genes(atac,min_cells=3)
sc.pp.filter_cells(atac,min_genes=200)
atac.var['mt'] = atac.var_names.str.startswith('MT-')
atac.var["ribo"] = atac.var_names.str.startswith(("RPS","RPL"))# hemoglobin genes
atac.var["hb"] = atac.var_names.str.contains("^HB[^(P)]")
sc.pp.calculate_qc_metrics(atac,qc_vars=['mt','ribo','hb'],percent_top=None,log1p=False,inplace=True)
sc.pl.violin(atac, ['n_genes_by_counts', 'total_counts', 'pct_counts_mt','pct_counts_hb','pct_counts_ribo'],
             jitter=0.4, multi_panel=True)

In [ ]:
atac = atac[atac.obs.n_genes_by_counts < 8000, :]
atac = atac[atac.obs.n_genes_by_counts > 300, :]

atac = atac[atac.obs.total_counts < 30000, :]


atac = atac[atac.obs.pct_counts_mt < 15, :]
#adatas = adatas[adatas.obs.pct_counts_hb < 2, :]
#adatas = adatas[adatas.obs.pct_counts_ribo < 40, :]
sc.pl.violin(atac, ['n_genes_by_counts', 'total_counts', 'pct_counts_mt'],
             jitter=0.4, multi_panel=True)


atac.layers['counts'] = atac.X.copy()

In [ ]:
atac.X = atac.layers["counts"].copy()
sc.experimental.pp.highly_variable_genes(
    atac,
    flavor='pearson_residuals',
    n_top_genes=3000,
    layer='counts',
    batch_key='sample'
)
sc.pp.normalize_total(atac,target_sum=1e4)
sc.pp.log1p(atac)

atac.layers['normalize'] = atac.X.copy()

## SCVI

In [ ]:
import scvi  # 变分自编码器框架（scVI/scANVI 模型）
import torch  # 底层深度学习后端 scvi需要用到GPU
print("device_count:", torch.cuda.device_count())
adata_hvg = atac[:, atac.var.highly_variable].copy()
scvi.model.SCVI.setup_anndata(adata_hvg, layer="counts", batch_key="sample")
model = scvi.model.SCVI(adata_hvg, n_layers=2, n_latent=30, gene_likelihood="nb")
model.train()
SCVI_LATENT_KEY = "X_scVI"
atac.obsm[SCVI_LATENT_KEY] = model.get_latent_representation()

sc.pp.neighbors(atac, use_rep=SCVI_LATENT_KEY,n_neighbors=15)
sc.tl.umap(atac)

## leiden

In [ ]:
import importlib
import helper.annotation_helper as ah
import helper.plot_helper as ph
importlib.reload(ah)
importlib.reload(ph)

In [ ]:
for resolution in [0.3,0.5,0.7,0.9,1.1]:
    sc.tl.leiden(atac,
                 key_added='clusters',
                 flavor="igraph",
                 n_iterations=-1,
                 resolution = resolution,
                 )
    sc.pl.umap(
            atac,
            color='clusters',
            legend_loc = 'on data'
        )

In [ ]:
sc.tl.leiden(atac,
                key_added='clusters',
                flavor="igraph",
                n_iterations=-1,#-1为直到收敛为最佳聚类，-2是默认值，迭代两次
                resolution = 0.5,#默认1，越高分的越多
                )
sc.pl.umap(
        atac,
        color='clusters',
        legend_loc = 'on data'
    )

In [ ]:
atac.X = atac.layers['normalize'].copy()

In [ ]:
ph.setGrid(True)

In [ ]:
marker_genes_dict = {
    "SFT_Tumor": ["IGF2", "LHX2", "NPW", "MGP", "ALDH1A1", "GRIA2", "PTGDS", "STAT6"],
    "Pericyte/SMC": ["RGS5", "NOTCH3", "PDGFRB", "MCAM", "NDUFA4L2", "ACTA2", "TAGLN"],
    "T/NK": ["CD3D", "CD3E", "TRAC", "TRBC2", "CCL5", "NKG7", "GNLY"],
    "Myeloid": ["PTPRC", "LYZ", "TYROBP", "AIF1", "LST1", "FCER1G", "CSF1R"],
    "Endo": ["PECAM1", "VWF", "FLT1", "PLVAP", "KDR", "RAMP2", "ENG"],
    "Mast": ["TPSAB1", "TPSB2", "CPA3", "KIT", "MS4A2"],
    "CAFs": ["LUM", "DCN", "COL1A1", "COL3A1", "SFRP2", "POSTN", "PDGFRA", "FAP", "COL6A1"],
    "Oligo": ["PLP1", "MAG", "MOBP", "MBP", "MOG"],
    "OPCs": ["PTPRZ1", "SOX10", "BCAN", "OLIG1", "OLIG2", "PDGFRA"]
}

sc.pl.dotplot(
    atac,
    marker_genes_dict,
    groupby="clusters",
    dendrogram=False,
    cmap="YlGnBu", # https://matplotlib.org/stable/users/explain/colors/colormaps.html
    #dot_max=0.6,     # 点最大直径比例（默认 1.0）
    #dot_min=0.1,     # 点最小直径比例（默认 0）
    #vmax=5,          # 显示的最大表达值，>3 的都显示为同一深红
    #vmin=0,           # 最小值（默认0）
    #save="_markers.pdf"
    standard_scale="var"


)